# 009 · Definite Integrals

| | |
|---|---|
| **Track** | Python for Calculus |
| **Level** | 3 — Integrals |
| **Prerequisites** | 002 and 008 |
| **Next** | 011 · Laplace Transform. Lesson 010 can already have been done after 007. |

**Place in the path.** A definite integral is a number. The Fundamental Theorem says it equals \(F(b) - F(a)\)
for any antiderivative \(F\). The constant \(C\) from lesson 008 cancels.
A Riemann sum from lesson 002's accumulation pattern estimates the same number, and the estimate improves as the partition gets finer.

## Learning objectives

By the end of this lesson you can:

- evaluate a definite integral exactly with SymPy
- recover that number from an antiderivative by the Fundamental Theorem
- estimate it with left, right, and midpoint Riemann sums
- explain signed area, including an integral of sine over a full period
- compare the exact value with `scipy.integrate.quad`


## 1. A polynomial, three ways

**See.** \(\int_0^1 x^2 \, dx\) is the area under \(y = x^2\) from 0 to 1.
An antiderivative is \(F(x) = x^3/3\). The Fundamental Theorem says the area is \(F(1) - F(0) = 1/3\).

**Do.** Compute it three ways.

1. `sp.integrate(f, (x, 0, 1))`
2. \(F(1) - F(0)\) from lesson 008's habit
3. `scipy.integrate.quad`, a numerical quadrature

All three must agree. The exact answers must be the rational \(1/3\), not a float that happens to look like it.


In [ ]:
import sympy as sp
from scipy.integrate import quad

x = sp.symbols("x")
integrand = x ** 2
exact = sp.integrate(integrand, (x, 0, 1))
antiderivative = sp.integrate(integrand, x)
by_ftc = antiderivative.subs(x, 1) - antiderivative.subs(x, 0)
numeric, estimated_error = quad(lambda t: t ** 2, 0, 1)

print("exact:", exact)
print("FTC:", by_ftc)
print("quad:", numeric, "estimated error", estimated_error)
assert exact == sp.Rational(1, 3)
assert sp.simplify(by_ftc - exact) == 0
assert abs(numeric - float(exact)) < 1e-10
print("three methods agree")


## 2. Sine on a half period, and signed area on a full period

**See.** \(\int_0^{\pi} \sin x \, dx = 2\). The arch above the axis has area 2.

On \([0, 2\pi]\) the positive arch and the negative arch cancel. The integral is 0.
That does not mean the curve didn't enclose area. It means the signed area is zero.
If you want geometric area, integrate \(|\sin x|\).

**Misconception.** An integral of zero means the function was zero.

**Correction.** It means the signed contributions cancelled. Plot the curve before you interpret the number.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

x = sp.symbols("x")
half = sp.integrate(sp.sin(x), (x, 0, sp.pi))
full = sp.integrate(sp.sin(x), (x, 0, 2 * sp.pi))
geometric = sp.integrate(sp.Abs(sp.sin(x)), (x, 0, 2 * sp.pi))
print("0 to pi:", half)
print("0 to 2pi, signed:", full)
print("0 to 2pi, absolute area:", geometric)
assert half == 2
assert full == 0
assert geometric == 4

xs = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(0, color="black", linewidth=0.8)
ax.fill_between(xs, np.sin(xs), where=np.sin(xs) >= 0, color="#1f4e79", alpha=0.35, label="positive area")
ax.fill_between(xs, np.sin(xs), where=np.sin(xs) < 0, color="#b85c38", alpha=0.35, label="negative area")
ax.plot(xs, np.sin(xs), color="black", linewidth=1.5)
ax.set_title(r"$\int_0^{2\pi}\sin x\,dx = 0$ because the two areas cancel")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.tight_layout()
plt.show()


## 3. A logarithm, by the Fundamental Theorem

**Do.**

\[
\int_1^{e} \ln x \, dx
\]

An antiderivative of \(\ln x\) is \(x \ln x - x\). Evaluate it from 1 to \(e\):

\[
(e \cdot 1 - e) - (0 - 1) = 1
\]

SymPy's definite integral must return 1. Differentiate the antiderivative to confirm it really is an antiderivative of \(\ln x\).


In [ ]:
import sympy as sp

x = sp.symbols("x", positive=True)
integrand = sp.log(x)
antiderivative = sp.integrate(integrand, x)
print("one antiderivative:", antiderivative)
assert sp.simplify(sp.diff(antiderivative, x) - integrand) == 0

exact = sp.integrate(integrand, (x, 1, sp.E))
by_ftc = sp.simplify(antiderivative.subs(x, sp.E) - antiderivative.subs(x, 1))
print("definite integral:", exact)
print("FTC:", by_ftc)
assert sp.simplify(exact - 1) == 0
assert sp.simplify(by_ftc - 1) == 0


## 4. Riemann sums, and the error going down

**See.** Cut \([0, 1]\) into \(n\) equal pieces of width \(\Delta x = 1/n\).
On each piece, replace the curve by a horizontal segment.

- left rule: height from the left endpoint
- right rule: height from the right endpoint
- midpoint rule: height from the center

For an increasing positive function such as \(x^2\) on \([0, 1]\), the left rule undershoots and the right rule overshoots.
The midpoint rule is usually much closer.

**Name.** This is the accumulation loop of lesson 002, written with NumPy so the sum is one line.
The exact value is \(1/3\). Print the error for \(n = 4, 20, 100\).


In [ ]:
import numpy as np

def riemann(f, a, b, n, rule):
    width = (b - a) / n
    if rule == "left":
        samples = np.linspace(a, b - width, n)
    elif rule == "right":
        samples = np.linspace(a + width, b, n)
    elif rule == "mid":
        samples = a + width * (np.arange(n) + 0.5)
    else:
        raise ValueError(rule)
    return float(np.sum(f(samples) * width))

def square(t):
    return t ** 2

exact = 1 / 3
print(f"{'n':>6} {'left':>12} {'right':>12} {'mid':>12} {'mid error':>12}")
previous_mid_error = None
for n in (4, 20, 100):
    left = riemann(square, 0, 1, n, "left")
    right = riemann(square, 0, 1, n, "right")
    mid = riemann(square, 0, 1, n, "mid")
    mid_error = abs(mid - exact)
    print(f"{n:6d} {left:12.8f} {right:12.8f} {mid:12.8f} {mid_error:12.8f}")
    assert left < exact < right
    if previous_mid_error is not None:
        assert mid_error < previous_mid_error
    previous_mid_error = mid_error
print("left undershoots, right overshoots, midpoint error decreases")


**Read the output.** At \(n = 4\) you can still see the gap. At \(n = 100\) the midpoint error is far below \(10^{-4}\).
The picture for \(n = 8\), right rule, makes the overshoot obvious: every rectangle covers a little triangle of curve.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

a, b, n = 0.0, 1.0, 8
width = (b - a) / n
edges = np.linspace(a, b, n + 1)
xs = np.linspace(a, b, 400)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, xs ** 2, color="#1f4e79", linewidth=2, label=r"$x^2$")
for left_edge, right_edge in zip(edges[:-1], edges[1:]):
    height = right_edge ** 2
    ax.add_patch(
        plt.Rectangle(
            (left_edge, 0),
            width,
            height,
            facecolor="#b85c38",
            edgecolor="black",
            alpha=0.35,
        )
    )
ax.set_title(r"Right Riemann sum for $\int_0^1 x^2\,dx$, $n = 8$. Exact value $1/3$.")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.tight_layout()
plt.show()


## Workshop — the three classic integrals

Confirm, exactly:

- \(\int_0^1 x^2 \, dx = 1/3\)
- \(\int_0^{\pi} \sin x \, dx = 2\)
- \(\int_1^{e} \ln x \, dx = 1\)

Then estimate the first one with 50 midpoint rectangles and demand an error below \(10^{-3}\).


In [ ]:
import numpy as np
import sympy as sp

x = sp.symbols("x", positive=True)
expected = [
    (x ** 2, 0, 1, sp.Rational(1, 3)),
    (sp.sin(x), 0, sp.pi, 2),
    (sp.log(x), 1, sp.E, 1),
]
for integrand, lower, upper, answer in expected:
    found = sp.integrate(integrand, (x, lower, upper))
    print(integrand, "from", lower, "to", upper, "=", found)
    assert sp.simplify(found - answer) == 0

n = 50
width = 1 / n
samples = width * (np.arange(n) + 0.5)
estimate = float(np.sum(samples ** 2 * width))
print("midpoint estimate:", estimate)
assert abs(estimate - 1 / 3) < 1e-3
print("workshop passed")


## Mistakes this lesson is meant to prevent

- Interpreting a zero integral as a zero function. Ask about signed area first.
- Comparing a Riemann sum to the exact value with `==`. The sum is an approximation until \(n \to \infty\).
- Forgetting that left and right sums swap their over/under behavior when the function is decreasing.
- Dropping the width \(\Delta x\). A sum of heights is not an area.

## What you now have, and what comes next

You can turn an integrand into a number and estimate that number from rectangles.
Lesson 011 writes the Laplace transform as one particular improper integral, from 0 to \(\infty\), of \(e^{-st} f(t)\).
Lesson 012 uses definite integrals only indirectly: a solution of a differential equation can be checked by substitution, and a numerical solver is a cousin of these approximations.
